In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import requests
import time
from Bio import SeqIO
from torch.utils.data import TensorDataset, DataLoader
import copy

import torch
import torch.nn as nn


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [2]:
labels = pd.read_csv("G-HumanEssential.tsv", sep="\t")

entrez_ids = labels["Gene ID"].astype(str).tolist()

url = "https://mygene.info/v3/gene"

mapping = {}

batch_size = 1000

for start in range(0, len(entrez_ids), batch_size):
    batch = entrez_ids[start:start + batch_size]

    response = requests.post(
        url,
        data={
            "ids": ",".join(batch),
            "scopes": "entrezgene",
            "fields": "entrezgene,ensembl.gene,symbol",
            "species": "human"
        }
    )

    response.raise_for_status()

    results = response.json()

    for result in results:
        if result.get("notfound"):
            continue

        entrez = str(result.get("entrezgene"))

        ensembl_data = result.get("ensembl")

        if isinstance(ensembl_data, dict):
            ensembl = ensembl_data.get("gene")

        elif isinstance(ensembl_data, list):
            ensembl = None

            for item in ensembl_data:
                if isinstance(item, dict) and item.get("gene"):
                    ensembl = item["gene"]
                    break

        else:
            ensembl = None

        if entrez and ensembl:
            mapping[entrez] = ensembl

    print(
        f"Processed {min(start + batch_size, len(entrez_ids)):,} "
        f"/ {len(entrez_ids):,} | "
        f"Mapped: {len(mapping):,}"
    )

    time.sleep(0.2)

print("\nFinished.")
print(f"Total labels: {len(entrez_ids):,}")
print(f"Successfully mapped: {len(mapping):,}")
print(f"Not mapped: {len(entrez_ids) - len(mapping):,}")

Processed 1,000 / 18,528 | Mapped: 992
Processed 2,000 / 18,528 | Mapped: 1,981
Processed 3,000 / 18,528 | Mapped: 2,975
Processed 4,000 / 18,528 | Mapped: 3,969
Processed 5,000 / 18,528 | Mapped: 4,957
Processed 6,000 / 18,528 | Mapped: 5,944
Processed 7,000 / 18,528 | Mapped: 6,932
Processed 8,000 / 18,528 | Mapped: 7,923
Processed 9,000 / 18,528 | Mapped: 8,916
Processed 10,000 / 18,528 | Mapped: 9,911
Processed 11,000 / 18,528 | Mapped: 10,901
Processed 12,000 / 18,528 | Mapped: 11,885
Processed 13,000 / 18,528 | Mapped: 12,879
Processed 14,000 / 18,528 | Mapped: 13,867
Processed 15,000 / 18,528 | Mapped: 14,856
Processed 16,000 / 18,528 | Mapped: 15,848
Processed 17,000 / 18,528 | Mapped: 16,838
Processed 18,000 / 18,528 | Mapped: 17,828
Processed 18,528 / 18,528 | Mapped: 18,352

Finished.
Total labels: 18,528
Successfully mapped: 18,352
Not mapped: 176


In [3]:
mapping_df = pd.DataFrame(
    list(mapping.items()),
    columns=["Gene ID", "Ensembl Gene ID"]
)

mapping_df["Gene ID"] = mapping_df["Gene ID"].astype(str)

labels_clean = labels.copy()
labels_clean["Gene ID"] = labels_clean["Gene ID"].astype(str)

labeled_genes = labels_clean.merge(
    mapping_df,
    on="Gene ID",
    how="inner"
)

print("Labeled genes:", len(labeled_genes))
print("\nClass distribution:")
print(labeled_genes[
    "Essentiality (determined from multiple datasets)"
].value_counts())

Labeled genes: 18351

Class distribution:
Essentiality (determined from multiple datasets)
Non-essential    16841
Essential         1510
Name: count, dtype: int64


In [4]:
fasta_path = "Homo_sapiens.GRCh38.cds.all.fa"

target_ensembl_ids = set(
    labeled_genes["Ensembl Gene ID"].astype(str)
)

matched_sequences = {}
matched_headers = {}

for record in SeqIO.parse(fasta_path, "fasta"):
    header = record.description

    gene_id = None

    for field in header.split():
        if field.startswith("gene:"):
            gene_id = field.split(":", 1)[1].split(".", 1)[0]
            break

    if gene_id in target_ensembl_ids:
        sequence = str(record.seq)

        if (
            gene_id not in matched_sequences
            or len(sequence) > len(matched_sequences[gene_id])
        ):
            matched_sequences[gene_id] = sequence
            matched_headers[gene_id] = header

print(f"Target genes:       {len(target_ensembl_ids):,}")
print(f"Genes found in FASTA: {len(matched_sequences):,}")
print(f"Genes not found:     {len(target_ensembl_ids - matched_sequences.keys()):,}")

Target genes:       18,349
Genes found in FASTA: 18,013
Genes not found:     336


In [5]:
label_column = "Essentiality (determined from multiple datasets)"

sequence_df = pd.DataFrame(
    [
        {
            "Ensembl Gene ID": gene_id,
            "sequence": sequence,
        }
        for gene_id, sequence in matched_sequences.items()
    ]
)

dataset = labeled_genes.merge(
    sequence_df,
    on="Ensembl Gene ID",
    how="inner"
)

dataset["label"] = (
    dataset[label_column] == "Essential"
).astype(int)

dataset = dataset[
    ["Gene ID", "Ensembl Gene ID", "sequence", "label"]
]

print(f"Final dataset size: {len(dataset):,}")

print(dataset["label"].value_counts())


Final dataset size: 18,015
label
0    16506
1     1509
Name: count, dtype: int64


In [7]:
from itertools import product

K = 3

kmers = [
    "".join(kmer)
    for kmer in product("ACGT", repeat=K)
]

kmer_to_idx = {kmer: i for i, kmer in enumerate(kmers)}

print(f"Number of possible {K}-mers:", len(kmers))
print(kmers)

Number of possible 3-mers: 64
['AAA', 'AAC', 'AAG', 'AAT', 'ACA', 'ACC', 'ACG', 'ACT', 'AGA', 'AGC', 'AGG', 'AGT', 'ATA', 'ATC', 'ATG', 'ATT', 'CAA', 'CAC', 'CAG', 'CAT', 'CCA', 'CCC', 'CCG', 'CCT', 'CGA', 'CGC', 'CGG', 'CGT', 'CTA', 'CTC', 'CTG', 'CTT', 'GAA', 'GAC', 'GAG', 'GAT', 'GCA', 'GCC', 'GCG', 'GCT', 'GGA', 'GGC', 'GGG', 'GGT', 'GTA', 'GTC', 'GTG', 'GTT', 'TAA', 'TAC', 'TAG', 'TAT', 'TCA', 'TCC', 'TCG', 'TCT', 'TGA', 'TGC', 'TGG', 'TGT', 'TTA', 'TTC', 'TTG', 'TTT']


In [8]:
def kmer_features(sequence, k = K):
    counts = np.zeros(4 ** k, dtype=np.float32)

    total = len(sequence) - k + 1

    if total <= 0:
        return counts

    for i in range(total):
        kmer = sequence[i:i+k]

        if kmer in kmer_to_idx:
            counts[kmer_to_idx[kmer]] += 1

    counts /= total

    return counts


X_kmer = np.vstack([
    kmer_features(seq, K)
    for seq in tqdm(dataset["sequence"], desc="Creating k-mer features")
])

y = dataset["label"].values

print("X shape:", X_kmer.shape)
print("y shape:", y.shape)

Creating k-mer features: 100%|██████████| 18015/18015 [00:05<00:00, 3450.04it/s]

X shape: (18015, 64)
y shape: (18015,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_kmer,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining distribution:")
print(pd.Series(y_train).value_counts())

print("\nTesting distribution:")
print(pd.Series(y_test).value_counts())

Training samples: 14412
Testing samples: 3603

Training distribution:
0    13205
1     1207
Name: count, dtype: int64

Testing distribution:
0    3301
1     302
Name: count, dtype: int64


In [ ]:
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train,
)

torch_scaler = StandardScaler()
X_tr_scaled = torch_scaler.fit_transform(X_tr)
X_val_scaled = torch_scaler.transform(X_val)
X_test_scaled_torch = torch_scaler.transform(X_test)

train_ds = TensorDataset(
    torch.tensor(X_tr_scaled, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.long),
)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test_scaled_torch, dtype=torch.float32, device=device)


class EssentialityMLP(nn.Module):
    def __init__(self, in_features=4**K, n_classes=2, dropout=0.3):
        super().__init__()

        self.fc1 = nn.Linear(in_features, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, n_classes)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        return self.fc3(x)


model = EssentialityMLP(in_features=X_tr.shape[1]).to(device)

print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

n_pos = int((y_tr == 1).sum())
n_neg = int((y_tr == 0).sum())

class_weights = torch.tensor(
    [len(y_tr) / (2 * n_neg), len(y_tr) / (2 * n_pos)],
    dtype=torch.float32,
    device=device,
)

print(f"\nTraining rows: {len(y_tr):,} (essential {n_pos:,} / non-essential {n_neg:,})")
print(f"Validation rows: {len(y_val):,} (essential {int(y_val.sum()):,})")
print(f"Class weights: non-essential {class_weights[0]:.3f} | essential {class_weights[1]:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)

MAX_EPOCHS = 300
PATIENCE = 30

best_ap = -np.inf
best_state = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()

    epoch_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()

        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(yb)

    epoch_loss /= len(train_ds)

    model.eval()

    with torch.no_grad():
        val_prob = torch.softmax(model(X_val_tensor), dim=1)[:, 1].cpu().numpy()

    val_ap = average_precision_score(y_val, val_prob)

    if val_ap > best_ap:
        best_ap = val_ap
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch {epoch:3d} | train loss {epoch_loss:.4f} | val PR-AUC {val_ap:.4f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Validation PR-AUC: {best_ap:.4f} (baseline {y_val.mean():.4f})")

Device: cpu
EssentialityMLP(
  (fc1): Linear(in_features=64, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=2, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
)
Trainable parameters: 16706

Training rows: 12,250 (essential 1,026 / non-essential 11,224)
Validation rows: 2,162 (essential 181)
Class weights: non-essential 0.546 | essential 5.970
epoch   1 | train loss 0.6379 | val PR-AUC 0.2024
epoch  10 | train loss 0.5433 | val PR-AUC 0.1889
epoch  20 | train loss 0.4906 | val PR-AUC 0.1992
epoch  30 | train loss 0.4430 | val PR-AUC 0.1839

Early stopping at epoch 36.

Best epoch: 6
Validation PR-AUC: 0.2077 (baseline 0.0837)


In [38]:
model.eval()

with torch.no_grad():
    test_prob = torch.softmax(model(X_test_tensor), dim=1)

    test_pred = test_prob.argmax(dim=1).cpu().numpy()
    test_prob_essential = test_prob[:, 1].cpu().numpy()

print("Classification report (softmax argmax):\n")
print(classification_report(y_test, test_pred, target_names=["Non-essential", "Essential"]))

print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))

print(f"\nROC-AUC: {roc_auc_score(y_test, test_prob_essential):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, test_prob_essential):.4f}")
print(f"PR-AUC baseline (prevalence): {y_test.mean():.4f}")

Classification report (softmax argmax):

               precision    recall  f1-score   support

Non-essential       0.96      0.74      0.84      3301
    Essential       0.19      0.65      0.29       302

     accuracy                           0.73      3603
    macro avg       0.57      0.70      0.56      3603
 weighted avg       0.89      0.73      0.79      3603

Confusion matrix:
[[2452  849]
 [ 106  196]]

ROC-AUC: 0.7792
PR-AUC:  0.2405
PR-AUC baseline (prevalence): 0.0838
